# Bangladesh: 0117_xee_tsharp

Static adaptation only; Earth Engine/data/asset execution not verified.

Original scientific assumptions and date windows retained; use the national workflow for QA improvements.

Review scale, missing data, units, and any training/calibration requirements before interpreting results.

Experimental model: use spatial/time holdout validation and independent Bangladesh ground truth; finer pixels do not prove added information.

May contain tutorial-specific assets or imported training features; inspect dependencies before execution.

In [ ]:
# Bangladesh adaptation: 0117_xee_tsharp
# Original author: Amirhossein Ahrari; see original attribution below and originals/.
from bdgee.core import initialize, boundaries
from bdgee.notebook import export_outputs, upload_local
initialize()  # Run `earthengine authenticate` first; project is configured in bdgee.
BD_COUNTRY, BD_DISTRICTS = boundaries()
BD_ROI = BD_COUNTRY.geometry()
BD_BBOX = [88.0, 20.5, 92.7, 26.7]  # search only, not analysis boundary
roi = geometry = border = iran = tehran = province = loc = point = BD_ROI
# Run from the repository root; all local outputs are isolated by source.
from pathlib import Path
import os
_REPO_ROOT = Path.cwd()
_OUTPUT_DIR = _REPO_ROOT / 'outputs' / 'urban' / '0117_xee_tsharp'
_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
os.chdir(_OUTPUT_DIR)


In [ ]:
"""GEE_0025_thermal_sharpening.ipynb


Tutorial Code by Amirhossein Ahrari
YouTube: https://www.youtube.com/@amirhosseinahrarigee
Tutorial Video: Google Earth Engine Tutorial-122: Thermal Sharpening using Machine Learning, in Python API (Xee)

This code is part of a tutorial series on Earth Engine programming techniques
presented on the Amirhossein Ahrari YouTube channel. You are free to use and modify
this code for academic and non-academic purposes. Don't forget to subscribe to
the Amirhossein Ahrari channel and follow the videos to support the instructor


"""
import ee
import geemap
import xarray as xr
import xee
None
None
map = geemap.Map(basemap='SATELLITE')
map
roi = BD_ROI
roi
lst = ee.ImageCollection('MODIS/061/MOD11A2').filterDate('2020', '2021').select('LST_Day_1km').mean().multiply(0.02).subtract(273.15)
ndvi = ee.ImageCollection('MODIS/061/MOD13Q1').filterDate('2020', '2021').select('NDVI').mean().multiply(0.0001)
stack = ee.Image.cat([lst, ndvi]).select([0, 1], ['lst', 'ndvi'])
stack
ds1km = xr.open_dataset(stack, engine='ee', crs='EPSG:4326', scale=0.01, geometry=BD_ROI)
ds250m = xr.open_dataset(stack, engine='ee', crs='EPSG:4326', scale=0.0025, geometry=BD_ROI)
ds1km
df1km = ds1km.to_dataframe()
df250m = ds250m.to_dataframe()
df1km
x = df1km[['ndvi']]
y = df1km['lst']
from sklearn.model_selection import train_test_split
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.05, random_state=42)
from sklearn.ensemble import RandomForestRegressor
model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(x_train, y_train)
y_pred = model.predict(x_test)
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)
print(f'RMSE:{rmse}')
print(f'R2:{r2}')
df250m['lst_sharp'] = model.predict(df250m[['ndvi']])
df250m
ds250 = df250m.to_xarray()
ds250
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 2, figsize=(8, 3))
plt.tight_layout(w_pad=3)
ds250.lst.plot(x='lon', y='lat', robust=True, cmap='hot', ax=ax[0])
ds250.lst_sharp.plot(x='lon', y='lat', robust=True, cmap='hot', ax=ax[1])
ax[0].set_title('1000 meters')
ax[1].set_title('250 meters')
plt.savefig('thsarp.png', dpi=360, bbox_inches='tight')


In [ ]:

# Review these task plans; task creation does not start exports.
os.chdir(_REPO_ROOT)
bd_tasks = export_outputs(globals(), 'urban', '0117_xee_tsharp', scale=1000)
# Set an appropriate native scale above before creating plans.
# for task in bd_tasks: task.start()
# upload_local(_OUTPUT_DIR, 'urban')  # explicit upload; requires Drive OAuth scope
